# ICH M11 Protocol Quality Judge

This notebook evaluates generated ICH M11 clinical trial protocol drafts using an LLM-as-Judge approach.

**Purpose:** Score protocols on quality dimensions and compare models side-by-side.

**How to use:**
1. Run the generation notebook (`protocol_gen_ich_m11`) first to produce protocols
2. Configure the parameters below (experiment path, model to evaluate, judge model)
3. Run all cells — the judge scores each protocol and logs assessments to MLflow traces

**Quality dimensions scored (1-5 scale):**
- **Completeness** — Required ICH M11 sections, objectives, criteria depth
- **Clinical Language** — Professional, regulatory-appropriate prose quality
- **Schema Adherence** — JSON structure matches expected ICH M11 schema
- **Traceability** — Source domain citations and data lineage
- **Overall** — Holistic demo-readiness assessment

In [0]:
# ============================================================
# Configuration — adjust these to control what gets evaluated
# ============================================================

# MLflow experiment where generation traces live
EXPERIMENT_PATH = "/Users/hfsl@novonordisk.com/ich-m11-protocol-gen/mlflow_ich_m11"

# Which generation model to evaluate (filters traces by tag)
# Set to None to evaluate ALL models in the experiment
MODEL_FILTER = "databricks-gpt-5-4-mini"

# Judge model — use a different model than the generator for unbiased scoring
JUDGE_MODEL = "databricks-claude-sonnet-4"
JUDGE_VERSION = "ich_m11_judge_v1"

# How many recent traces to evaluate (per model)
MAX_TRACES = 10

# Delta table fallback (used if traces don't contain full protocol content)
PROTOCOL_TABLE = "sandbox.hfsl.m11_protocols"

# Set to a specific run_id to evaluate only that run, or None for latest
RUN_ID_FILTER = None

In [0]:
import json
import mlflow
from databricks.sdk import WorkspaceClient

mlflow.set_experiment(EXPERIMENT_PATH)
mlflow_client = mlflow.MlflowClient()

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

# --- Locate traces to evaluate ---
experiment = mlflow.get_experiment_by_name(EXPERIMENT_PATH)
assert experiment, f"Experiment not found: {EXPERIMENT_PATH}"

# Build filter for trace search
trace_filter = ""
if MODEL_FILTER:
    trace_filter = f"tag.model = '{MODEL_FILTER}'"
if RUN_ID_FILTER:
    run_clause = f"tag.`mlflow.run_id` = '{RUN_ID_FILTER}'"
    trace_filter = f"{trace_filter} AND {run_clause}" if trace_filter else run_clause

print(f"🔍 Searching traces in experiment: {EXPERIMENT_PATH}")
print(f"   Filter: {trace_filter or '(all)'}")

traces = mlflow_client.search_traces(
    experiment_ids=[experiment.experiment_id],
    filter_string=trace_filter,
    order_by=["timestamp_ms DESC"],
    max_results=MAX_TRACES,
)

print(f"   Found {len(traces)} trace(s)")

# --- Extract protocol content from traces ---
protocols_to_evaluate = []

for trace in traces:
    trace_id = trace.info.trace_id
    tags = {t.key: t.value for t in (trace.info.tags or [])}
    study_id = tags.get("study_id", "UNKNOWN")
    model_name = tags.get("model", "unknown")

    # Skip traces already judged (avoid re-scoring)
    if tags.get("judge/overall") and not RUN_ID_FILTER:
        print(f"   ⏭️  Skipping {study_id} (already judged: overall={tags['judge/overall']})")
        continue

    # Extract protocol JSON from trace output
    protocol_json = None
    try:
        # Trace outputs contain the full OpenAI response
        outputs = trace.data.spans[0].outputs if trace.data.spans else None
        if outputs and isinstance(outputs, dict):
            choices = outputs.get("choices", [])
            if choices:
                content = choices[0].get("message", {}).get("content", "")
                if isinstance(content, str):
                    parsed = json.loads(content)
                    protocol_json = parsed.get("protocol", parsed)
    except (json.JSONDecodeError, KeyError, IndexError, TypeError):
        pass

    if protocol_json:
        protocols_to_evaluate.append({
            "trace_id": trace_id,
            "study_id": study_id,
            "model": model_name,
            "protocol": protocol_json,
            "source": "mlflow_trace",
        })
    else:
        print(f"   ⚠️  Could not extract protocol from trace {trace_id} ({study_id})")

# --- Fallback: read from Delta table if no protocols extracted from traces ---
if not protocols_to_evaluate:
    print(f"\n📦 Falling back to Delta table: {PROTOCOL_TABLE}")
    rows = spark.sql(f"""
        SELECT
            id,
            c.value:protocol AS protocol,
            c.value:protocol:title_page:sponsor_protocol_identifier::string AS study_id,
            model,
            mlflow_run_id
        FROM {PROTOCOL_TABLE} p, LATERAL variant_explode(p.content) AS c
        WHERE model = '{MODEL_FILTER or ''}' OR '{MODEL_FILTER or ''}' = ''
        ORDER BY id DESC
        LIMIT {MAX_TRACES}
    """).collect()

    for row in rows:
        protocols_to_evaluate.append({
            "trace_id": None,
            "study_id": row["study_id"] or "UNKNOWN",
            "model": row["model"] or "unknown",
            "protocol": json.loads(row["protocol"]) if isinstance(row["protocol"], str) else row["protocol"],
            "source": "delta_table",
        })

print(f"\n✅ {len(protocols_to_evaluate)} protocol(s) ready for judge evaluation")
for p in protocols_to_evaluate:
    print(f"   📄 {p['study_id']} (model={p['model']}, source={p['source']})")

In [0]:
# ============================================================
# ICH M11 Quality Evaluation Rubric
# ============================================================

JUDGE_SYSTEM_PROMPT = """
You are an expert clinical protocol reviewer evaluating AI-generated ICH M11 clinical trial protocol drafts.

Score the protocol on each dimension using a 1-5 scale:
- 5: Excellent — meets or exceeds regulatory submission quality
- 4: Good — minor improvements needed, largely complete
- 3: Adequate — usable but has notable gaps or weaknesses
- 2: Poor — significant issues that require major revision
- 1: Unacceptable — missing critical content or fundamentally flawed

Return your evaluation as valid JSON with this exact structure:
{
  "scores": {
    "completeness": <1-5>,
    "clinical_language": <1-5>,
    "schema_adherence": <1-5>,
    "traceability": <1-5>,
    "overall": <1-5>
  },
  "rationale": {
    "completeness": "<1-2 sentence justification>",
    "clinical_language": "<1-2 sentence justification>",
    "schema_adherence": "<1-2 sentence justification>",
    "traceability": "<1-2 sentence justification>",
    "overall": "<1-2 sentence overall assessment>"
  },
  "strengths": ["<strength 1>", "<strength 2>"],
  "improvements": ["<improvement 1>", "<improvement 2>"]
}
"""

JUDGE_USER_TEMPLATE = """
Evaluate this ICH M11 protocol draft for study: {study_id}

Scoring dimensions:
1. **Completeness**: Are all required ICH M11 sections present? Are there ≥12 protocol sections? Are objectives (2-3 primary, 3-4 secondary), inclusion/exclusion criteria (6-8 each), and visit descriptions substantive?
2. **Clinical Language**: Is the prose professional, factual, scientifically neutral, and regulatory-appropriate? Does it read like a senior medical writer authored it (not bullet points or placeholders)?
3. **Schema Adherence**: Does the JSON structure follow the expected ICH M11 schema? Are all required fields populated? Are nested objects well-formed?
4. **Traceability**: Are source domains cited correctly? Are SDTM TS and TV references used appropriately? Are inferred or missing values flagged?
5. **Overall**: Holistic assessment considering all dimensions and demo-readiness.

Protocol JSON to evaluate:
{protocol_json}
"""


def judge_protocol(protocol_json: dict, study_id: str, client, judge_model: str) -> dict:
    """
    Score a single protocol using the LLM judge.
    Returns parsed scores dict or error dict.
    """
    # Truncate to avoid token limits (keep first 15k chars of JSON)
    protocol_str = json.dumps(protocol_json, indent=None)[:15000]

    user_message = JUDGE_USER_TEMPLATE.format(
        study_id=study_id,
        protocol_json=protocol_str
    )

    try:
        response = client.chat.completions.create(
            model=judge_model,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
            temperature=0.0,
            max_tokens=1500,
        )
        raw = response.choices[0].message.content
        # Strip markdown code fences if present
        if raw.strip().startswith("```"):
            raw = raw.strip().split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(raw)
    except Exception as e:
        print(f"  ⚠️ Judge failed for {study_id}: {e}")
        return {"scores": {}, "rationale": {}, "error": str(e)}


print(f"✅ Judge configured: {JUDGE_MODEL} ({JUDGE_VERSION})")
print(f"   Scoring dimensions: completeness, clinical_language, schema_adherence, traceability, overall")

In [0]:
# ============================================================
# Execute Judge on All Protocols
# ============================================================

import time

print(f"\n🧑‍⚖️ Running ICH M11 Quality Judge ({JUDGE_MODEL}) on {len(protocols_to_evaluate)} protocol(s)...\n")

evaluation_results = []

for i, entry in enumerate(protocols_to_evaluate, 1):
    study_id = entry["study_id"]
    trace_id = entry["trace_id"]
    protocol = entry["protocol"]

    print(f"  ⏳ [{i}/{len(protocols_to_evaluate)}] Judging: {study_id} (model={entry['model']}) ...")

    t0 = time.time()
    judge_result = judge_protocol(protocol, study_id, client, JUDGE_MODEL)
    judge_latency = time.time() - t0

    scores = judge_result.get("scores", {})

    # --- Log assessments back to the generation trace (if available) ---
    if scores and trace_id:
        for dimension, score in scores.items():
            rationale = judge_result.get("rationale", {}).get(dimension, "")
            mlflow_client.set_trace_tag(trace_id, f"judge/{dimension}", str(score))
            mlflow_client.set_trace_tag(trace_id, f"judge/{dimension}/rationale", rationale[:250])

        mlflow_client.set_trace_tag(trace_id, "judge/model", JUDGE_MODEL)
        mlflow_client.set_trace_tag(trace_id, "judge/version", JUDGE_VERSION)

    if scores:
        print(f"  ✅ {study_id}: overall={scores.get('overall')}, "
              f"completeness={scores.get('completeness')}, "
              f"clinical_language={scores.get('clinical_language')}, "
              f"schema={scores.get('schema_adherence')}, "
              f"traceability={scores.get('traceability')} "
              f"({judge_latency:.1f}s)")
    else:
        print(f"  ❌ {study_id}: Judge returned no scores")

    # Store full result
    evaluation_results.append({
        "study_id": study_id,
        "model": entry["model"],
        "trace_id": trace_id,
        "source": entry["source"],
        "judge_model": JUDGE_MODEL,
        "judge_latency_s": round(judge_latency, 2),
        "scores": scores,
        "rationale": judge_result.get("rationale", {}),
        "strengths": judge_result.get("strengths", []),
        "improvements": judge_result.get("improvements", []),
    })

print(f"\n✅ Judge evaluation complete: {len(evaluation_results)} protocol(s) scored")

In [0]:
# ============================================================
# Evaluation Summary & Model Comparison
# ============================================================

import pandas as pd

if not evaluation_results:
    print("⚠️ No evaluation results. Run the judge cell first.")
else:
    # --- Per-study scores ---
    summary_rows = []
    for r in evaluation_results:
        row = {
            "study_id": r["study_id"],
            "model": r["model"],
            "judge_model": r["judge_model"],
            "completeness": r["scores"].get("completeness"),
            "clinical_language": r["scores"].get("clinical_language"),
            "schema_adherence": r["scores"].get("schema_adherence"),
            "traceability": r["scores"].get("traceability"),
            "overall": r["scores"].get("overall"),
            "strengths": "; ".join(r.get("strengths", [])),
            "improvements": "; ".join(r.get("improvements", [])),
        }
        summary_rows.append(row)

    eval_df = pd.DataFrame(summary_rows)

    print("📊 Judge Scores by Study & Model")
    print("=" * 80)
    display(eval_df[["study_id", "model", "completeness", "clinical_language",
                     "schema_adherence", "traceability", "overall"]])

    # --- Aggregate by model ---
    print("\n🏆 Model Average Scores (higher is better)")
    print("-" * 60)
    score_cols = ["completeness", "clinical_language", "schema_adherence", "traceability", "overall"]
    agg = eval_df.groupby("model")[score_cols].mean().round(2).sort_values("overall", ascending=False)
    display(agg)

    # --- Best model recommendation ---
    if len(agg) > 1:
        best_model = agg["overall"].idxmax()
        best_score = agg.loc[best_model, "overall"]
        print(f"\n🥇 Recommended model: {best_model} (overall={best_score})")
    elif len(agg) == 1:
        print(f"\n💡 Only one model evaluated ({agg.index[0]}). Run with a different MODEL_FILTER to compare.")

    # --- Historical comparison from MLflow runs ---
    print("\n🔍 Historical Runs with Judge Scores (from MLflow)")
    print("-" * 60)
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="params.judge_model != ''",
        order_by=["start_time DESC"],
        max_results=20,
    )
    if not runs.empty:
        display_cols = ["params.model", "params.judge_model", "params.num_studies"]
        metric_cols = [c for c in runs.columns if c.startswith("metrics.judge/") and "/mean" in c]
        available = [c for c in display_cols + metric_cols if c in runs.columns]
        if available:
            hist_df = runs[available].rename(
                columns=lambda c: c.replace("params.", "").replace("metrics.judge/", "")
            )
            display(hist_df.head(10))
        else:
            print("  No judge metrics found in run history yet.")
    else:
        print("  No historical runs with judge evaluation found yet.")
        print("  Tip: Log judge metrics to runs using the generation notebook's enhanced tracing.")

    print("\n" + "=" * 80)
    print("💡 To compare models:")
    print("   1. Run protocol_gen_ich_m11 with model='databricks-gpt-5-4-mini'")
    print("   2. Run protocol_gen_ich_m11 with model='databricks-claude-sonnet-4'")
    print("   3. Re-run this notebook with MODEL_FILTER=None to score all")
    print("   4. The summary above will show scores side-by-side")